In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold

df = pd.read_parquet(
    "../Results/cleaned_cic_ids2017.parquet"
)

print(df.shape)
display(df.head())

(2313810, 81)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,SourceFile,AttackType,BinaryLabel
0,6,4,2,0,12,0,6,6,6.00000,0.000000,...,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,Benign,0
1,6,1,2,0,12,0,6,6,6.00000,0.000000,...,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,Benign,0
2,6,3,2,0,12,0,6,6,6.00000,0.000000,...,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,Benign,0
3,6,1,2,0,12,0,6,6,6.00000,0.000000,...,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,Benign,0
4,6,609,7,4,484,414,233,0,69.14286,111.967896,...,0,0,0.0,0.0,0,0,Benign,Benign-Monday-no-metadata.parquet,Benign,0


In [2]:
import os
import glob
import pandas as pd

path = r"D:\ANM\Topic 30\Dataset Full"  # Đường dẫn tới thư mục chứa file .parquet
files = glob.glob(os.path.join(path, "*.parquet"))

dataframes = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dataframes, ignore_index=True)

# Xác định target

In [3]:
TARGET = "BinaryLabel"

META_COLUMNS = [
    "Label",
    "AttackType",
    "BinaryLabel",
    "SourceFile"
]

print("Target:", TARGET)
print("Metadata columns:", META_COLUMNS)

Target: BinaryLabel
Metadata columns: ['Label', 'AttackType', 'BinaryLabel', 'SourceFile']


# Encoding categorical features

In [4]:
import glob
import os
import pandas as pd

categorical_columns = df_all.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['Label']


# Encode categorical
- Không encode target ở đây vì target đã là 0/1.
- Nếu dataset của bạn đã có Protocol dưới dạng số thì cell này sẽ không đụng vào nó.

In [5]:
encoders = {}

for col in categorical_columns:

    if col in META_COLUMNS:
        continue

    encoder = LabelEncoder()

    df[col] = encoder.fit_transform(
        df[col].astype(str)
    )

    encoders[col] = encoder

print("Encoded categorical columns:")
print(list(encoders.keys()))

Encoded categorical columns:
[]


# Feature Engineering

In [7]:
def find_column(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None


fwd_packets = find_column(
    df,
    ["Total Fwd Packets", "Tot Fwd Pkts"]
)

bwd_packets = find_column(
    df,
    ["Total Backward Packets", "Tot Bwd Pkts"]
)

fwd_length = find_column(
    df,
    ["Total Length of Fwd Packets", "TotLen Fwd Pkts"]
)

bwd_length = find_column(
    df,
    ["Total Length of Bwd Packets", "TotLen Bwd Pkts"]
)

print("Forward packets :", fwd_packets)
print("Backward packets:", bwd_packets)
print("Forward bytes   :", fwd_length)
print("Backward bytes  :", bwd_length)

Forward packets : Total Fwd Packets
Backward packets: Total Backward Packets
Forward bytes   : None
Backward bytes  : None


# Tạo ratio features

In [8]:
if fwd_packets and bwd_packets:

    df["Fwd_Bwd_Packet_Ratio"] = (
        df[fwd_packets] /
        (df[bwd_packets] + 1)
    )


if fwd_length and bwd_length:

    df["Fwd_Bwd_Byte_Ratio"] = (
        df[fwd_length] /
        (df[bwd_length] + 1)
    )


print(
    "Feature engineering completed."
)

Feature engineering completed.


# Feature list

In [9]:
X = df.drop(
    columns=META_COLUMNS,
    errors="ignore"
)

y = df[TARGET]

# Only numeric features
X = X.select_dtypes(include=np.number)

print("Number of features before selection:", X.shape[1])

Number of features before selection: 78


# Remove invalid values

In [10]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

valid_rows = X.notna().all(axis=1)

X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2313810, 78)
y shape: (2313810,)


# Variance feature selection

In [11]:
variance_selector = VarianceThreshold(
    threshold=0
)

X_selected = variance_selector.fit_transform(X)

selected_features = X.columns[
    variance_selector.get_support()
]

X_selected = pd.DataFrame(
    X_selected,
    columns=selected_features,
    index=X.index
)

print(
    "Features before selection:",
    X.shape[1]
)

print(
    "Features after selection:",
    X_selected.shape[1]
)

Features before selection: 78
Features after selection: 70


# Feature names

In [12]:
print("Selected features:")
print("="*20)
for feature in selected_features:
    print(feature)

Selected features:
Protocol
Flow Duration
Total Fwd Packets
Total Backward Packets
Fwd Packets Length Total
Bwd Packets Length Total
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Bytes/s
Flow Packets/s
Flow IAT Mean
Flow IAT Std
Flow IAT Max
Flow IAT Min
Fwd IAT Total
Fwd IAT Mean
Fwd IAT Std
Fwd IAT Max
Fwd IAT Min
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Bwd IAT Min
Fwd PSH Flags
Fwd URG Flags
Fwd Header Length
Bwd Header Length
Fwd Packets/s
Bwd Packets/s
Packet Length Min
Packet Length Max
Packet Length Mean
Packet Length Std
Packet Length Variance
FIN Flag Count
SYN Flag Count
RST Flag Count
PSH Flag Count
ACK Flag Count
URG Flag Count
CWE Flag Count
ECE Flag Count
Down/Up Ratio
Avg Packet Size
Avg Fwd Segment Size
Avg Bwd Segment Size
Subflow Fwd Packets
Subflow Fwd Bytes
Subflow Bwd Packets
Subflow Bwd Bytes
Init Fwd Win Bytes
Init Bw

# Correlation analysis

In [13]:
corr_matrix = X_selected.corr().abs()

upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

high_corr_features = [
    column
    for column in upper.columns
    if any(upper[column] > 0.95)
]

print(
    "Highly correlated features:",
    len(high_corr_features)
)

print(high_corr_features[:50])

Highly correlated features: 22
['Total Backward Packets', 'Bwd Packets Length Total', 'Fwd Packet Length Std', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd IAT Total', 'Fwd IAT Max', 'Fwd Packets/s', 'Packet Length Std', 'SYN Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Avg Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Idle Mean', 'Idle Max', 'Idle Min']


# Remove highly correlated features

In [14]:
X_final = X_selected.drop(
    columns=high_corr_features,
    errors="ignore"
)

print(
    "Final number of features:",
    X_final.shape[1]
)

Final number of features: 48


# Final feature list

In [15]:
feature_columns = X_final.columns.tolist()

print("FINAL FEATURES:")
print("=" * 20)

for i, feature in enumerate(feature_columns, 1):
    print(f"{i:3d}. {feature}")

FINAL FEATURES:
  1. Protocol
  2. Flow Duration
  3. Total Fwd Packets
  4. Fwd Packets Length Total
  5. Fwd Packet Length Max
  6. Fwd Packet Length Min
  7. Fwd Packet Length Mean
  8. Bwd Packet Length Max
  9. Bwd Packet Length Min
 10. Flow Bytes/s
 11. Flow Packets/s
 12. Flow IAT Mean
 13. Flow IAT Std
 14. Flow IAT Max
 15. Flow IAT Min
 16. Fwd IAT Mean
 17. Fwd IAT Std
 18. Fwd IAT Min
 19. Bwd IAT Total
 20. Bwd IAT Mean
 21. Bwd IAT Std
 22. Bwd IAT Max
 23. Bwd IAT Min
 24. Fwd PSH Flags
 25. Fwd URG Flags
 26. Fwd Header Length
 27. Bwd Header Length
 28. Bwd Packets/s
 29. Packet Length Min
 30. Packet Length Max
 31. Packet Length Mean
 32. Packet Length Variance
 33. FIN Flag Count
 34. RST Flag Count
 35. PSH Flag Count
 36. ACK Flag Count
 37. URG Flag Count
 38. Down/Up Ratio
 39. Init Fwd Win Bytes
 40. Init Bwd Win Bytes
 41. Fwd Act Data Packets
 42. Fwd Seg Size Min
 43. Active Mean
 44. Active Std
 45. Active Max
 46. Active Min
 47. Idle Std
 48. Fwd_Bwd_Pac

# Save feature data

In [16]:
os.makedirs("../Models", exist_ok=True)
os.makedirs("../Results", exist_ok=True)

X_final.to_parquet(
    "../Results/X_final.parquet",
    index=True
)

y.to_frame("BinaryLabel").to_parquet(
    "../Results/y_final.parquet",
    index=True
)

joblib.dump(
    feature_columns,
    "../Models/feature_columns.joblib"
)

joblib.dump(
    variance_selector,
    "../Models/feature_selector.joblib"
)

print("Feature pipeline saved.")

Feature pipeline saved.
